## Library

In [ ]:
!pip install PySastrawi nltk openpyxl

import pandas as pd
import re
import nltk
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.6/210.6 kB 2.7 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

## Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

df = pd.read_csv('comments_therealgitasav_7196636282427116805_20260311_205831.csv')

print(f'   Jumlah baris : {len(df)}')
print(f'   Kolom        : {list(df.columns)}')
df.head()

Saving comments_therealgitasav_7196636282427116805_20260311_205831.csv to comments_therealgitasav_7196636282427116805_20260311_205831.csv
   Jumlah baris : 71
   Kolom        : ['no', 'username', 'display_name', 'text', 'date', 'likes', 'reply_to', 'is_reply']


,no,username,display_name,text,date,likes,reply_to,is_reply
0,1,akhwat_jalang,Akhwat_jalang,couple yg begini yg bikin aing baper dan perca...,2023-02-05T19:50:26.000Z,1943,NaN,False
1,2,lesspalmlatte,lesspalmlatte,bahagia selalu kak 🥰,2023-02-05T18:56:23.000Z,135,NaN,False
2,3,partohaps,paul partohap,im the luckiest,2023-02-05T18:54:19.000Z,390,NaN,False
3,4,prisla.a,madam prisla🔮Tarot&Affirmasi,lucu bgtt 🥹🫶🏻,2023-02-05T19:30:18.000Z,475,NaN,False
4,5,sopsopy7,Sopi sopiyah17,yang aku inginkan suatu hari nanti,2023-02-05T19:40:00.000Z,85,NaN,False


## Init

In [ ]:
factory_stem = StemmerFactory()
id_stemmer   = factory_stem.create_stemmer()
en_lemmatizer = WordNetLemmatizer()

factory_sw   = StopWordRemoverFactory()
id_stopwords = set(factory_sw.get_stop_words())
en_stopwords = set(stopwords.words('english'))
extra_stopwords = {
    'yg','ga','gak','nggak','aja','kak','loh','lah','sih','deh','dong','nih',
    'ya','wkwk','haha','hehe','gw','gue','lo','lu','aku','kamu','dia','nya',
    'tapi','tp','krn','utk','jg','juga','dan','atau','jd','kalau','klo','emg',
    'banget','bgt','sm','sama','uda','sudah'
}
all_stopwords = id_stopwords.union(en_stopwords).union(extra_stopwords)
common_en_words = set(stopwords.words('english')).union({
    'love','like','good','great','beautiful','happy','nice','best','better',
    'really','very','much','always','every','im','its','people','think','know'
})


## Emote


In [ ]:
emoji_positif = {
    '🥰','😍','❤️','🤍','🧡','💛','💚','💙','💜','🖤','🤎','💕','💞','💓','💗','💖','💝',
    '😊','😄','😁','🥹','🫶','🫶🏻','👏','🙌','🙏','✨','🌟','⭐','💫','🔥','🌈',
    '😂','🤣','😭','😢','🥺','💗','🌸','🌺','🌻','🌼','🍀','🎉','🎊','👑','💪','🤗',
}
emoji_negatif = {
    '😠','😡','🤬','👎','💔','😤','🙄','😒','😑','😐','🤮','🤢','😞','😟','😣',
    '😖','😫','😩','💀','☠️','🗑️',
}

def emoji_to_sentiment(text):
    """Konversi emoji ke token sentimen sebelum cleaning."""
    if pd.isna(text): return ''
    text = str(text)
    for e in emoji_positif: text = text.replace(e, ' EMO_POS ')
    for e in emoji_negatif: text = text.replace(e, ' EMO_NEG ')
    return text

print('Fungsi konversi emoji siap!')


Fungsi konversi emoji siap!


## Text Cleaning

In [ ]:
def clean_text(text):
    if pd.isna(text) or text == '': return ''
    text = emoji_to_sentiment(text)        # Emote
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text_cleaned'] = df['text'].apply(clean_text)
print('Pembersihan teks selesai!')
df[['text', 'text_cleaned']].head(10)

Pembersihan teks selesai!


,text,text_cleaned
0,couple yg begini yg bikin aing baper dan perca...,couple yg begini yg bikin aing baper dan perca...
1,bahagia selalu kak 🥰,bahagia selalu kak emo_pos
2,im the luckiest,im the luckiest
3,lucu bgtt 🥹🫶🏻,lucu bgtt emo_pos emo_pos
4,yang aku inginkan suatu hari nanti,yang aku inginkan suatu hari nanti
5,bahagia terus kak git🥰,bahagia terus kak git emo_pos
6,knpaaa gemasss sekali 🥺💗,knpaaa gemasss sekali emo_pos emo_pos
7,lucuuu,lucuuu
8,bahagia selalu kakak kakak,bahagia selalu kakak kakak
9,stay healthy 🥰,stay healthy emo_pos


## Tokenization

In [ ]:
def tokenize(text):
    if not text: return []
    return word_tokenize(text)

df['tokens'] = df['text_cleaned'].apply(tokenize)
print('Tokenisasi selesai!')
df[['text_cleaned', 'tokens']].head(10)

Tokenisasi selesai!


,text_cleaned,tokens
0,couple yg begini yg bikin aing baper dan perca...,"[couple, yg, begini, yg, bikin, aing, baper, d..."
1,bahagia selalu kak emo_pos,"[bahagia, selalu, kak, emo_pos]"
2,im the luckiest,"[im, the, luckiest]"
3,lucu bgtt emo_pos emo_pos,"[lucu, bgtt, emo_pos, emo_pos]"
4,yang aku inginkan suatu hari nanti,"[yang, aku, inginkan, suatu, hari, nanti]"
5,bahagia terus kak git emo_pos,"[bahagia, terus, kak, git, emo_pos]"
6,knpaaa gemasss sekali emo_pos emo_pos,"[knpaaa, gemasss, sekali, emo_pos, emo_pos]"
7,lucuuu,[lucuuu]
8,bahagia selalu kakak kakak,"[bahagia, selalu, kakak, kakak]"
9,stay healthy emo_pos,"[stay, healthy, emo_pos]"


## Stopword Removal

In [ ]:
def remove_stopwords(tokens):

    return [t for t in tokens if t in ('emo_pos','emo_neg') or (t not in all_stopwords and len(t) > 1)]

df['tokens_no_stopword'] = df['tokens'].apply(remove_stopwords)
print(' Stopword removal selesai!')
df[['tokens', 'tokens_no_stopword']].head(10)

 Stopword removal selesai!


,tokens,tokens_no_stopword
0,"[couple, yg, begini, yg, bikin, aing, baper, d...","[couple, bikin, aing, baper, percaya, pernikah..."
1,"[bahagia, selalu, kak, emo_pos]","[bahagia, emo_pos]"
2,"[im, the, luckiest]","[im, luckiest]"
3,"[lucu, bgtt, emo_pos, emo_pos]","[lucu, bgtt, emo_pos, emo_pos]"
4,"[yang, aku, inginkan, suatu, hari, nanti]",[]
5,"[bahagia, terus, kak, git, emo_pos]","[bahagia, git, emo_pos]"
6,"[knpaaa, gemasss, sekali, emo_pos, emo_pos]","[knpaaa, gemasss, emo_pos, emo_pos]"
7,[lucuuu],[lucuuu]
8,"[bahagia, selalu, kakak, kakak]","[bahagia, kakak, kakak]"
9,"[stay, healthy, emo_pos]","[stay, healthy, emo_pos]"


## Stemming & Lemmatization

In [ ]:
def is_english_word(word):
    return word in common_en_words

def stem_and_lemmatize(tokens):
    result = []
    for token in tokens:
        if token in ('emo_pos', 'emo_neg'):
            result.append(token)
        elif is_english_word(token):
            result.append(en_lemmatizer.lemmatize(token))
        else:
            result.append(id_stemmer.stem(token))
    return result

df['tokens_stemmed'] = df['tokens_no_stopword'].apply(stem_and_lemmatize)
df['text_final']     = df['tokens_stemmed'].apply(lambda x: ' '.join(x))

df[['tokens_no_stopword', 'tokens_stemmed', 'text_final']].head(10)

,tokens_no_stopword,tokens_stemmed,text_final
0,"[couple, bikin, aing, baper, percaya, pernikah...","[couple, bikin, aing, baper, percaya, nikah, b...",couple bikin aing baper percaya nikah bahagia
1,"[bahagia, emo_pos]","[bahagia, emo_pos]",bahagia emo_pos
2,"[im, luckiest]","[im, luckiest]",im luckiest
3,"[lucu, bgtt, emo_pos, emo_pos]","[lucu, bgtt, emo_pos, emo_pos]",lucu bgtt emo_pos emo_pos
4,[],[],
5,"[bahagia, git, emo_pos]","[bahagia, git, emo_pos]",bahagia git emo_pos
6,"[knpaaa, gemasss, emo_pos, emo_pos]","[knpaaa, gemasss, emo_pos, emo_pos]",knpaaa gemasss emo_pos emo_pos
7,[lucuuu],[lucuuu],lucuuu
8,"[bahagia, kakak, kakak]","[bahagia, kakak, kakak]",bahagia kakak kakak
9,"[stay, healthy, emo_pos]","[stay, healthy, emo_pos]",stay healthy emo_pos


## Pre-Processing Result

In [ ]:
print(f'Total baris          : {len(df)}')
print(f'Rata-rata token awal : {df["tokens"].apply(len).mean():.2f}')
print(f'Rata-rata token akhir: {df["tokens_stemmed"].apply(len).mean():.2f}')
df[['text', 'text_cleaned', 'text_final']].head(10)

Total baris          : 71
Rata-rata token awal : 5.66
Rata-rata token akhir: 3.56


,text,text_cleaned,text_final
0,couple yg begini yg bikin aing baper dan perca...,couple yg begini yg bikin aing baper dan perca...,couple bikin aing baper percaya nikah bahagia
1,bahagia selalu kak 🥰,bahagia selalu kak emo_pos,bahagia emo_pos
2,im the luckiest,im the luckiest,im luckiest
3,lucu bgtt 🥹🫶🏻,lucu bgtt emo_pos emo_pos,lucu bgtt emo_pos emo_pos
4,yang aku inginkan suatu hari nanti,yang aku inginkan suatu hari nanti,
5,bahagia terus kak git🥰,bahagia terus kak git emo_pos,bahagia git emo_pos
6,knpaaa gemasss sekali 🥺💗,knpaaa gemasss sekali emo_pos emo_pos,knpaaa gemasss emo_pos emo_pos
7,lucuuu,lucuuu,lucuuu
8,bahagia selalu kakak kakak,bahagia selalu kakak kakak,bahagia kakak kakak
9,stay healthy 🥰,stay healthy emo_pos,stay healthy emo_pos


## Auto-labeling (Positif / Negatif)

Metode: **Lexicon-Based** dengan perbaikan:
- ✅ Emoji dikonversi ke token sentimen (EMO_POS/EMO_NEG) sebelum dihapus
- ✅ Kamus kata diperluas untuk konteks komentar media sosial
- ✅ Komentar pendek (≤3 kata) yang tidak jelas → default **Positif** sesuai konteks video

In [ ]:
kata_positif = {
    'support','supportive','mendukung','dukung','setuju','agree','respect',
    'bangga','proud','salut','berani','brave','bahagia','happy','senang',
    'suka','like','love','luv','sayang','bagus','good','great','nice','best',
    'amazing','wonderful','cute','sweet','sweeett','lucu','lucuuu','lucuuuuuu',
    'lucukkkk','gemas','gemasss','emesh','uwu','lucky','luckiest','blessed',
    'semangat','sukses','sehat','aamiin','amiin','masyaallah','alhamdulillah',
    'manifesting','manifestingg','jannah','berkah','couple','pasangan',
    'favorit','kesayangan','forever','pernikahan','nikah','emo_pos',
    'mau','pengen','pgnnn','ingin','someday','goals','terharu','baper','omooo',
}
kata_negatif = {
    'tidak setuju','gak setuju','kontra','salah','wrong','disagree',
    'haram','dosa','berdosa','tobat','kasihan','rugi','menyesal','aneh',
    'weird','gila','egois','egoistis','selfish','payah','buruk','jelek','bad',
    'menyedihkan','memalukan','malu','negatif','haters','sewot','nyinyir',
    'depresi','tekanan','patriarki','emo_neg',
}

def labeli(text, text_asli):
    if not text or text.strip() == '':
        return 'Positif'
    text_lower = text.lower()
    skor_pos = sum(1 for k in kata_positif if k in text_lower)
    skor_neg = sum(1 for k in kata_negatif if k in text_lower)
    skor_pos += text_lower.count('emo_pos') * 2  # Bobot lebih untuk emoji
    skor_neg += text_lower.count('emo_neg') * 2
    if skor_pos > skor_neg: return 'Positif'
    elif skor_neg > skor_pos: return 'Negatif'
    else:
        words = [w for w in text.split() if w not in ('emo_pos','emo_neg')]
        if len(words) == 0: return 'Positif'
        if len(words) <= 3: return 'Positif'
        return 'Negatif'

df['Label'] = df.apply(lambda r: labeli(r['text_final'], r['text']), axis=1)

print('Pelabelan selesai!')
print(f'\nDistribusi Label:')
print(df['Label'].value_counts())
df[['text', 'text_final', 'Label']].head(15)

Pelabelan selesai!

Distribusi Label:
Label
Positif    63
Negatif     8
Name: count, dtype: int64


,text,text_final,Label
0,couple yg begini yg bikin aing baper dan perca...,couple bikin aing baper percaya nikah bahagia,Positif
1,bahagia selalu kak 🥰,bahagia emo_pos,Positif
2,im the luckiest,im luckiest,Positif
3,lucu bgtt 🥹🫶🏻,lucu bgtt emo_pos emo_pos,Positif
4,yang aku inginkan suatu hari nanti,,Positif
5,bahagia terus kak git🥰,bahagia git emo_pos,Positif
6,knpaaa gemasss sekali 🥺💗,knpaaa gemasss emo_pos emo_pos,Positif
7,lucuuu,lucuuu,Positif
8,bahagia selalu kakak kakak,bahagia kakak kakak,Positif
9,stay healthy 🥰,stay healthy emo_pos,Positif


In [ ]:
for _, row in df.head(10).iterrows():
    print(f'Teks Asli  : {row["text"]}')
    print(f'Teks Final : {row["text_final"]}')
    print(f'Label      : {row["Label"]}')
    print()

Teks Asli  : couple yg begini yg bikin aing baper dan percaya kalau pernikahan masih sangat mungkin semembahagiakan itu
Teks Final : couple bikin aing baper percaya nikah bahagia
Label      : Positif

Teks Asli  : bahagia selalu kak 🥰
Teks Final : bahagia emo_pos
Label      : Positif

Teks Asli  : im the luckiest
Teks Final : im luckiest
Label      : Positif

Teks Asli  : lucu bgtt 🥹🫶🏻
Teks Final : lucu bgtt emo_pos emo_pos
Label      : Positif

Teks Asli  : yang aku inginkan suatu hari nanti
Teks Final : 
Label      : Positif

Teks Asli  : bahagia terus kak git🥰
Teks Final : bahagia git emo_pos
Label      : Positif

Teks Asli  : knpaaa gemasss sekali 🥺💗
Teks Final : knpaaa gemasss emo_pos emo_pos
Label      : Positif

Teks Asli  : lucuuu
Teks Final : lucuuu
Label      : Positif

Teks Asli  : bahagia selalu kakak kakak
Teks Final : bahagia kakak kakak
Label      : Positif

Teks Asli  : stay healthy 🥰
Teks Final : stay healthy emo_pos
Label      : Positif



## Export

In [ ]:
BIRU_TUA='1F4E79'; HIJAU='E2EFDA'; MERAH_MUDA='FCE4D6'; PUTIH='FFFFFF'; ABU='F2F2F2'; BIRU_MUDA='D6E4F0'
border = Border(
    left=Side(style='thin',color='CCCCCC'), right=Side(style='thin',color='CCCCCC'),
    top=Side(style='thin',color='CCCCCC'),  bottom=Side(style='thin',color='CCCCCC')
)

wb = openpyxl.Workbook()
ws = wb.active
ws.title = 'Pelabelan'

headers    = ['No', 'Username', 'Display Name', 'Text (Asli)', 'Text (Preprocessing)', 'Label']
col_widths = [6, 25, 25, 40, 40, 14]
for ci, (h, w) in enumerate(zip(headers, col_widths), 1):
    cell = ws.cell(row=1, column=ci, value=h)
    cell.font=Font(name='Arial',bold=True,color=PUTIH,size=11)
    cell.fill=PatternFill('solid',fgColor=BIRU_TUA)
    cell.alignment=Alignment(horizontal='center',vertical='center')
    cell.border=border
    ws.column_dimensions[get_column_letter(ci)].width = w
ws.row_dimensions[1].height = 30

for i, row in df.iterrows():
    r=i+2; label=row['Label']; bg=ABU if r%2==0 else PUTIH
    lbl_fill=PatternFill('solid',fgColor=HIJAU if label=='Positif' else MERAH_MUDA)
    lbl_font=Font(name='Arial',size=10,bold=True,color='375623' if label=='Positif' else '9C0006')
    row_data=[row.get('no',i+1),row.get('username',''),row.get('display_name',''),row.get('text',''),row.get('text_final',''),label]
    for ci, val in enumerate(row_data, 1):
        cell=ws.cell(row=r,column=ci,value=val); cell.border=border
        if ci==6:
            cell.font=lbl_font; cell.fill=lbl_fill
            cell.alignment=Alignment(horizontal='center',vertical='center')
        else:
            cell.font=Font(name='Arial',size=10); cell.fill=PatternFill('solid',fgColor=bg)
            cell.alignment=Alignment(horizontal='left',vertical='top',wrap_text=True)
    ws.row_dimensions[r].height=45

ws2=wb.create_sheet('Ringkasan')
ws2['A1']='Ringkasan Pelabelan — TikTok Comments (therealgitasav)'
ws2['A1'].font=Font(name='Arial',bold=True,size=13,color=BIRU_TUA)
ws2.merge_cells('A1:C1')
for ci,h in enumerate(['Label','Jumlah','Persentase'],1):
    c=ws2.cell(row=3,column=ci,value=h)
    c.font=Font(name='Arial',bold=True,color=PUTIH,size=11)
    c.fill=PatternFill('solid',fgColor=BIRU_TUA)
    c.alignment=Alignment(horizontal='center')
    ws2.column_dimensions[get_column_letter(ci)].width=22

total=len(df); counts=df['Label'].value_counts()
for ri,(lbl,cnt) in enumerate(counts.items(),4):
    for ci,val in enumerate([lbl,cnt,f'{cnt/total*100:.1f}%'],1):
        c=ws2.cell(row=ri,column=ci,value=val)
        c.font=Font(name='Arial',size=11); c.alignment=Alignment(horizontal='center'); c.border=border
tr=len(counts)+4
for ci,val in enumerate(['Total',total,'100%'],1):
    c=ws2.cell(row=tr,column=ci,value=val)
    c.font=Font(name='Arial',bold=True,size=11)
    c.fill=PatternFill('solid',fgColor=BIRU_MUDA)
    c.alignment=Alignment(horizontal='center'); c.border=border

output_file='Tugas_Pelabelan_Dataset2_TikTok_therealgitasav_Nama_NIM.xlsx'
wb.save(output_file)
print(f'✅ File disimpan: {output_file}')
print(f'   Total   : {total}')
print(f'   Positif : {counts.get("Positif",0)}')
print(f'   Negatif : {counts.get("Negatif",0)}')

✅ File disimpan: Tugas_Pelabelan_Dataset2_TikTok_therealgitasav_Nama_NIM.xlsx
   Total   : 71
   Positif : 63
   Negatif : 8


## Download

In [ ]:
from google.colab import files
files.download('Tugas_Pelabelan_Dataset2_TikTok_therealgitasav_Nama_NIM.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>